In [1]:
import pandas as pd
import sqlite3

# Loading cleaned dataset
df = pd.read_csv('/content/clean_enriched_claims.csv')

# Connecting to SQLite
conn = sqlite3.connect(':memory:')
df.to_sql('claims', conn, index=False, if_exists='replace')

pd.read_sql("SELECT COUNT(*) AS total_rows FROM claims", conn)

,total_rows
0,1188


In [2]:
''' KPI 1: Calculate the total cost, headcount, and average cost per employee
    for each employer group (fair per-capita comparison)'''

query1 = """
SELECT
    EmployerGroup,
    COUNT(DISTINCT PatientID) AS employees_covered,
    COUNT(*) AS total_claims,
    ROUND(SUM(ClaimAmount), 2) AS total_cost,
    ROUND(SUM(ClaimAmount) * 1.0 / COUNT(DISTINCT PatientID), 2) AS cost_per_employee
FROM claims
GROUP BY EmployerGroup
ORDER BY cost_per_employee DESC
"""
result1 = pd.read_sql(query1, conn)
result1

,EmployerGroup,employees_covered,total_claims,total_cost,cost_per_employee
0,Employer_15,71,71,394204.14,5552.17
1,Employer_12,69,69,375732.32,5445.40
2,Employer_03,74,74,394116.09,5325.89
3,Employer_02,68,68,359063.79,5280.35
4,Employer_01,82,82,430942.93,5255.40
5,Employer_13,65,65,338786.36,5212.10
6,Employer_16,55,55,282446.27,5135.39
7,Employer_14,57,57,292578.64,5132.96
8,Employer_04,64,64,326800.47,5106.26
9,Employer_08,74,74,374659.08,5062.96


In [3]:
'''
KPI 2: Frequency vs. severity by claim type :
(this tells us why costs are high ,lots of small claims, or a few huge ones):
'''

query2 = """
SELECT
    ClaimType,
    COUNT(*) AS claim_count,
    ROUND(AVG(ClaimAmount), 2) AS avg_claim_amount,
    ROUND(SUM(ClaimAmount), 2) AS total_cost,
    ROUND(SUM(ClaimAmount) * 100.0 / (SELECT SUM(ClaimAmount) FROM claims), 1) AS pct_of_total_cost
FROM claims
GROUP BY ClaimType
ORDER BY total_cost DESC
"""
result2 = pd.read_sql(query2, conn)
result2

,ClaimType,claim_count,avg_claim_amount,total_cost,pct_of_total_cost
0,Inpatient,313,4892.04,1531207.66,25.7
1,Outpatient,297,5109.65,1517565.44,25.5
2,Routine,306,4756.19,1455395.48,24.5
3,Emergency,272,5317.75,1446427.16,24.3


In [4]:
'''KPI 3: High-cost claimant concentration'''

query3 = """
WITH patient_totals AS (
    SELECT
        PatientID,
        SUM(ClaimAmount) AS patient_total_cost
    FROM claims
    GROUP BY PatientID
),
ranked AS (
    SELECT
        *,
        NTILE(10) OVER (ORDER BY patient_total_cost DESC) AS decile
    FROM patient_totals
)
SELECT
    decile,
    COUNT(*) AS num_patients,
    ROUND(SUM(patient_total_cost), 2) AS decile_total_cost,
    ROUND(SUM(patient_total_cost) * 100.0 / (SELECT SUM(patient_total_cost) FROM patient_totals), 1) AS pct_of_total_cost
FROM ranked
GROUP BY decile
ORDER BY decile
"""
result3 = pd.read_sql(query3, conn)
result3

,decile,num_patients,decile_total_cost,pct_of_total_cost
0,1,119,1129934.48,19.0
1,2,119,1009116.15,17.0
2,3,119,880716.57,14.8
3,4,119,759264.29,12.8
4,5,119,652185.52,11.0
5,6,119,535321.80,9.0
6,7,119,421656.03,7.1
7,8,119,307550.99,5.2
8,9,118,185020.52,3.1
9,10,118,69829.39,1.2


In [5]:
'''
Checking the  cost gap between the highest-cost
group (Employer_15) and lowest-cost group (Employer_07) by
comparing their plan tier mix and average claim amounts.
'''

query4 = """
SELECT
    EmployerGroup,
    PlanTier,
    COUNT(*) AS claims,
    ROUND(AVG(ClaimAmount), 2) AS avg_claim_amount
FROM claims
WHERE EmployerGroup IN ('Employer_15', 'Employer_07')
GROUP BY EmployerGroup, PlanTier
ORDER BY EmployerGroup, PlanTier
"""
result4 = pd.read_sql(query4, conn)
result4

,EmployerGroup,PlanTier,claims,avg_claim_amount
0,Employer_07,Bronze,24,3543.90
1,Employer_07,Gold,16,4291.66
2,Employer_07,Silver,25,4652.20
3,Employer_15,Bronze,18,5764.68
4,Employer_15,Gold,23,5578.50
5,Employer_15,Silver,30,5404.48
